# Student Health Risk — Playground Series S6E7

**Task**: classificazione multi-classe (`fit` / `at-risk` / `unhealthy`)
**Metrica**: **balanced accuracy** → ogni classe pesa uguale, indipendentemente dal supporto
**Dataset**: 690.088 righe, 7 feature numeriche + 6 categoriche, null su tutte le feature

## Log delle versioni
| ver | descrizione | OOF BA (argmax) | OOF BA (prior-corr, t*) | LB |
|-----|-------------|-----------------|--------------------------|-----|
| v1  | LightGBM baseline, NaN crudi, prior-correction | 0.87626 | **0.94938** (t=1.00) | **0.94591** ✓ |
| v2  | v1 + `bmi_is_missing` (ablation) | *(in corso)* | *(in corso)* | — |

## Decisioni chiave (e perché)
1. **Nessuna imputation nella baseline**: LightGBM gestisce i NaN nativamente (impara la direzione dello split); le categoriche trattano il NaN come categoria propria. Qualsiasi imputation dovrà *battere* questa baseline in ablation.
2. **Prior-correction**: con sbilanciamento 86/8/6 e balanced accuracy, l'argmax del posteriore è subottimale. Si usa `argmax P(c|x) / prior^t`. Sweep su t: **t*=1.00** → probabilità ben calibrate (stesso risultato di S6E6).
3. **Diagnostica missingness** (vedi sotto): missingness ≈ MCAR ovunque tranne `bmi`, il cui tasso di null varia 4x tra `unhealthy` (0.71%) e `fit` (2.86%) → unico indicatore candidato come feature (v2).
4. **Co-missingness `exercise_duration`/`diet_type`**: entrambe 6.901 null, ma overlap osservato 72 ≈ 69 atteso sotto indipendenza → stessi conteggi marginali, nessun pattern strutturato.

**Ancoraggio OOF↔LB verificato** (v1): gap 0.0035, fisiologico (public LB = frazione del test; t* scelto su OOF → lieve ottimismo). OOF su 690k righe = stima più affidabile del public LB.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

import kagglehub
kagglehub.competition_download('playground-series-s6e7')
BASE = '/kaggle/input/competitions/playground-series-s6e7'

SEED = 42
N_FOLDS = 5
print('Setup ✓')

In [ ]:
train = pd.read_csv(f'{BASE}/train.csv')
test  = pd.read_csv(f'{BASE}/test.csv')
print(train.shape, test.shape)

## 1. EDA essenziale

Target fortemente sbilanciato: `at-risk` 85.9%, `unhealthy` 8.4%, `fit` 5.8%.
Con la balanced accuracy, il campo di battaglia sono le due classi minoritarie.

In [ ]:
display(train['health_condition'].value_counts())
display(train['health_condition'].value_counts(normalize=True).round(4))
train.isnull().sum()

## 2. Diagnostica della missingness

Tre domande, in ordine:
1. i null di colonne con conteggi identici cadono sulle stesse righe? (co-missingness)
2. la missingness è correlata tra colonne? (pattern strutturati)
3. la missingness correla col **target**? (→ feature gratis)

**Esito** (prima esecuzione): missingness ≈ MCAR ovunque, unica eccezione `bmi`
(null: 0.71% in `unhealthy`, 2.08% in `at-risk`, 2.86% in `fit`).

In [ ]:
# 1. co-missingness: exercise_duration e diet_type hanno entrambe 6901 null
overlap = (train['exercise_duration'].isnull() & train['diet_type'].isnull()).sum()
expected = train['exercise_duration'].isnull().sum() * train['diet_type'].isnull().sum() / len(train)
print(f'overlap osservato: {overlap} | atteso sotto indipendenza: {expected:.0f}')

# 2. correlazione della missingness tra colonne
msno = train.drop(columns=['id', 'health_condition']).isnull()
plt.figure(figsize=(9, 7))
sns.heatmap(msno.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlazione della missingness')
plt.show()

# 3. frazione di null per classe (la domanda da un milione di punti)
null_by_class = train.groupby('health_condition')[msno.columns].apply(lambda g: g.isnull().mean()).T
display(null_by_class.round(4))

## 3. Preprocessing minimale

- NaN numerici: lasciati crudi (LightGBM li gestisce nativamente)
- Categoriche: cast a `category` (il NaN diventa di fatto una categoria)
- Target: LabelEncoder → **usare sempre `le.classes_` per rimappare le predizioni**, mai a mano

In [ ]:
TARGET = 'health_condition'
cat_cols = train.select_dtypes('object').columns.drop(TARGET).tolist()

le = LabelEncoder()
y = le.fit_transform(train[TARGET])
print('ordine classi:', dict(enumerate(le.classes_)))

def make_X(df, add_bmi_missing=False):
    X = df.drop(columns=['id', TARGET], errors='ignore').copy()
    if add_bmi_missing:
        X['bmi_is_missing'] = X['bmi'].isnull().astype(int)
    for c in cat_cols:
        X[c] = X[c].astype('category')
    return X

X      = make_X(train)
X_test = make_X(test)
priors = np.bincount(y) / len(y)
print('priors:', priors.round(4))

## 4. Pipeline CV riusabile

Un'unica funzione per baseline e ablation: stessi fold, stesso seed → confronti puliti.
Restituisce OOF, modelli di fold e BA per fold.

In [ ]:
LGB_PARAMS = dict(
    objective='multiclass', num_class=3,
    n_estimators=2000, learning_rate=0.05,
    num_leaves=63, random_state=SEED,
    verbosity=-1,
)

def run_cv(X, y, params=LGB_PARAMS, n_folds=N_FOLDS, seed=SEED, label=''):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = np.zeros((len(X), 3))
    models, fold_scores = [], []
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        model = lgb.LGBMClassifier(**params)
        model.fit(X.iloc[tr], y[tr],
                  eval_set=[(X.iloc[va], y[va])],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = model.predict_proba(X.iloc[va])
        models.append(model)
        ba = balanced_accuracy_score(y[va], oof[va].argmax(1))
        fold_scores.append(ba)
        print(f'[{label}] fold {fold}: BA = {ba:.5f}')
    print(f'[{label}] OOF BA (argmax): {balanced_accuracy_score(y, oof.argmax(1)):.5f}')
    return oof, models, fold_scores

def prior_sweep(oof, y, priors, ts=np.arange(0.0, 1.55, 0.25)):
    results = {}
    for t in ts:
        results[round(float(t), 2)] = balanced_accuracy_score(y, (oof / priors**t).argmax(1))
    for t, ba in results.items():
        print(f't={t:.2f}: BA = {ba:.5f}')
    t_star = max(results, key=results.get)
    print(f'--> t* = {t_star:.2f}, BA = {results[t_star]:.5f}')
    return t_star, results

## 5. v1 — baseline

Risultati prima esecuzione: fold BA 0.874–0.879 (CV stabilissima), OOF argmax **0.87626**,
prior-correction **+7.3 punti** → **0.94938** a t*=1.00.

In [ ]:
oof_v1, models_v1, _ = run_cv(X, y, label='v1')
t_star, sweep_v1 = prior_sweep(oof_v1, y, priors)

### Sweep fine intorno a t* (opzionale, per i perfezionisti)

In [ ]:
_ = prior_sweep(oof_v1, y, priors, ts=np.arange(0.85, 1.16, 0.05))

## 6. Submission (v1 + prior-correction)

Media delle `predict_proba` dei 5 modelli di fold, correzione con i prior del **train**, argmax,
rimappatura con `le.classes_`.

⚠️ Prima submission = **ancoraggio OOF ↔ LB**: se il LB ≈ 0.949, la macchina è verificata.

In [ ]:
T_SUB = 1.0  # aggiornare se lo sweep fine trova di meglio

test_proba = np.mean([m.predict_proba(X_test) for m in models_v1], axis=0)
test_pred  = le.classes_[(test_proba / priors**T_SUB).argmax(1)]

sub = pd.DataFrame({'id': test['id'], TARGET: test_pred})
sub.to_csv('submission.csv', index=False)
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()

## 7. v2 — ablation: `bmi_is_missing`

Ipotesi dalla diagnostica: il tasso di null di `bmi` varia 4x tra `unhealthy` e `fit`.
Stessi fold, stesso seed della v1 → il delta è attribuibile solo alla feature.

**Criterio di adozione**: delta OOF (dopo prior-correction) positivo e coerente sui fold.
Altrimenti si scarta, senza rimpianti.

In [ ]:
X_v2      = make_X(train, add_bmi_missing=True)
X_test_v2 = make_X(test,  add_bmi_missing=True)

oof_v2, models_v2, _ = run_cv(X_v2, y, label='v2')
t_star_v2, sweep_v2 = prior_sweep(oof_v2, y, priors)

# confronto onesto: delta a parita' di fold e seed, dopo prior-correction
ba_v1 = max(sweep_v1.values())
ba_v2 = max(sweep_v2.values())
print(f'\nv1: {ba_v1:.5f} | v2: {ba_v2:.5f} | delta: {ba_v2 - ba_v1:+.5f}')

### Verdetto v2

- delta > +0.0005 e coerente sui fold → si adotta, la submission v2 usa `models_v2` e `X_test_v2`
- delta ≈ 0 o negativo → si scarta e si passa a Optuna (obiettivo: **BA dopo prior-correction**, con `enqueue_trial` dei parametri baseline come anti-regressione)